In [1]:
import torch
import torch.nn.functional as F
import torch.nn as nn
# import torch.optim as optim
# import torch_optimizer as optim
import torch.nn.init as init
import adabound
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

from sys import stdout
from tqdm import tqdm
from tqdm import trange
import seaborn as sns

import time
from datetime import datetime as dt

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/zorocrit/Control_Nuclear_Spins/master/C13spin/testdata.csv?token=GHSAT0AAAAAAB5HMDTAB4MPQRTCFHHQLYUWZA5N5QA')
# df

In [3]:
# y = df[['x', 'N', 'z']]
# # y

y = df[['Xtau', 'XN', 'Ztau', 'ZN']]
y

,Xtau,XN,Ztau,ZN
0,2.959641,10.0,0.176380,1.0
1,4.245750,7.0,0.625249,22.0
2,3.594517,10.0,1.129403,1.0
3,0.479873,16.0,0.059788,1.0
4,2.861716,13.0,0.197018,4.0
...,...,...,...,...
5851,2.768226,10.0,0.179441,13.0
5852,1.721697,19.0,0.808594,1.0
5853,3.170644,10.0,0.256000,1.0
5854,0.326812,10.0,0.019688,13.0


In [4]:
X = df[['Al', 'Ap']]
# X

In [5]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.10, random_state=100)

In [6]:
Xdata = list(np.array(X_train.values.tolist()))
print(Xdata.__len__())

5270


In [7]:
ydata = list(np.array(y_train.values.tolist()))
# print(ydata)

In [8]:
torch.manual_seed(1)

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)
print(device)

cpu


In [10]:
x = torch.FloatTensor(Xdata).to(device)
y = torch.FloatTensor(ydata).to(device)

c:\Users\Administrator\anaconda3\lib\site-packages\ipykernel_launcher.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  C:\b\abs_bao0hdcrdh\croot\pytorch_1675190257512\work\torch\csrc\utils\tensor_new.cpp:204.)
  """Entry point for launching an IPython kernel.


In [11]:
num_data = Xdata.__len__()

num_epoch = 1000000

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
torch.manual_seed(1)

In [16]:
W = torch.zeros((2, 1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# optimizer 설정

optimizer = adabound.AdaBound([W, b], lr=0.01, final_lr=0.1)

In [18]:
nb_epochs = 20000
for epoch in range(nb_epochs + 1):

    # H(x) 계산
    # 편향 b는 브로드 캐스팅되어 각 샘플에 더해집니다.
    hypothesis = x.matmul(W) + b

    # cost 계산
    cost = torch.mean((hypothesis - y) ** 2)

    # cost로 H(x) 개선
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    print('Epoch {:4d}/{} hypothesis: {} Cost: {:.6f}'.format(
        epoch, nb_epochs, hypothesis.squeeze().detach(), cost.item()
    ))

Epoch    0/20000 hypothesis: tensor([0.1503, 0.2802, 0.1276,  ..., 0.1048, 0.2523, 0.2606]) Cost: 48.382565
Epoch    1/20000 hypothesis: tensor([0.1511, 0.2815, 0.1283,  ..., 0.1053, 0.2535, 0.2619]) Cost: 48.374561
Epoch    2/20000 hypothesis: tensor([0.1518, 0.2829, 0.1289,  ..., 0.1059, 0.2547, 0.2632]) Cost: 48.366550
Epoch    3/20000 hypothesis: tensor([0.1525, 0.2843, 0.1295,  ..., 0.1064, 0.2560, 0.2644]) Cost: 48.358543
Epoch    4/20000 hypothesis: tensor([0.1533, 0.2856, 0.1301,  ..., 0.1069, 0.2572, 0.2657]) Cost: 48.350552
Epoch    5/20000 hypothesis: tensor([0.1540, 0.2870, 0.1307,  ..., 0.1074, 0.2584, 0.2670]) Cost: 48.342548
Epoch    6/20000 hypothesis: tensor([0.1547, 0.2884, 0.1314,  ..., 0.1079, 0.2596, 0.2682]) Cost: 48.334560
Epoch    7/20000 hypothesis: tensor([0.1555, 0.2897, 0.1320,  ..., 0.1084, 0.2609, 0.2695]) Cost: 48.326569
Epoch    8/20000 hypothesis: tensor([0.1562, 0.2911, 0.1326,  ..., 0.1089, 0.2621, 0.2708]) Cost: 48.318581
Epoch    9/20000 hypothesis: